# TrOCR Fine-Tuning - Kaggle (T4 x2)

Fine-tunes `microsoft/trocr-base-handwritten` on a custom dataset.

**Setup before running:**
1. Settings -> Accelerator -> **GPU T4 x2**
2. Settings -> Internet -> **On** (needed to download the base model)
3. Add your dataset (with `manifest.csv` + `train/ val/ test/` folders) and set `DATA_ROOT` below.

Includes the decoder-start-token fix, size-aware config (5k / 10k), encoder freezing,
warmup + LR scheduler, gradient clipping, mixed precision, multi-GPU (DataParallel), and **checkpointing by validation CER**.

In [ ]:
# 1. (Optional) ensure a recent transformers. Kaggle usually has it preinstalled.
# Uncomment if you hit version issues.
# !pip install -q -U transformers

print('[OK] cell ready')  # --status--

In [3]:
# 2. Imports
import os, time, math, warnings, logging
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')
logging.disable(logging.WARNING)

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from PIL import Image
from tqdm.auto import tqdm
from transformers import (TrOCRProcessor, VisionEncoderDecoderModel,
                          get_linear_schedule_with_warmup)

print('[OK] Imports')  # --status--

In [4]:
# 3. GPU check
print('CUDA available :', torch.cuda.is_available())
n_gpu = torch.cuda.device_count()
print('GPU count      :', n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CUDA available : True
GPU count      : 2
  GPU 0: Tesla T4  15.6 GB
  GPU 1: Tesla T4  15.6 GB


In [6]:
# 4. CONFIG  --  pick a profile that matches your dataset size
MODEL_NAME = 'microsoft/trocr-base-handwritten'

# Auto-detect the folder containing manifest.csv under /kaggle/input.
# Works regardless of the dataset name or whether you zipped the parent folder.
DATA_ROOT = None
for _root, _dirs, _files in os.walk('/kaggle/input'):
    if 'manifest.csv' in _files:
        DATA_ROOT = _root
        break
if DATA_ROOT is None:
    DATA_ROOT = '/kaggle/input/your-dataset-name'  # fallback: set manually if not found
    print('WARNING: manifest.csv not auto-found. Set DATA_ROOT manually.')
print('DATA_ROOT =', DATA_ROOT)
OUTPUT_DIR = '/kaggle/working/trocr-finetuned'

MANIFEST_CSV  = os.path.join(DATA_ROOT, 'manifest.csv')
TRAIN_IMG_DIR = os.path.join(DATA_ROOT, 'train')
VAL_IMG_DIR   = os.path.join(DATA_ROOT, 'val')
TEST_IMG_DIR  = os.path.join(DATA_ROOT, 'test')

# Size-aware presets. BATCH_SIZE is the TOTAL batch, split across the 2 T4s by DataParallel.
PROFILES = {
    '5k': dict(EPOCHS=4, BATCH_SIZE=16, GRAD_ACCUM_STEPS=2,
               DECODER_LR=2e-5, ENCODER_LR=1e-5, WARMUP_RATIO=0.10,
               FREEZE_ENCODER=True,  UNFREEZE_AT_EPOCH=1,
               EARLY_STOP_PATIENCE=2, MAX_LABEL_LENGTH=40),
    '10k': dict(EPOCHS=3, BATCH_SIZE=16, GRAD_ACCUM_STEPS=4,
               DECODER_LR=3e-5, ENCODER_LR=1.5e-5, WARMUP_RATIO=0.05,
               FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
               EARLY_STOP_PATIENCE=2, MAX_LABEL_LENGTH=40),
    '20k': dict(EPOCHS=2, BATCH_SIZE=16, GRAD_ACCUM_STEPS=6,
               DECODER_LR=3e-5, ENCODER_LR=1.5e-5, WARMUP_RATIO=0.05,
               FREEZE_ENCODER=False, UNFREEZE_AT_EPOCH=0,
               EARLY_STOP_PATIENCE=1, MAX_LABEL_LENGTH=40),
}

PROFILE = '10k'          # <-- set to '5k', '10k', or '20k'
cfg = PROFILES[PROFILE]
EVAL_BATCH_SIZE = 32     # generation batch for validation/test (eval is on 1 GPU)
NUM_WORKERS = 2
print('Profile:', PROFILE)
for k, v in cfg.items():
    print(f'  {k:18}: {v}')

DATA_ROOT = /kaggle/input/datasets/mirolim998/synthetic-handwritten-datasets-10000-samples/dataset_001
Profile: 10k
  EPOCHS            : 3
  BATCH_SIZE        : 16
  GRAD_ACCUM_STEPS  : 4
  DECODER_LR        : 3e-05
  ENCODER_LR        : 1.5e-05
  WARMUP_RATIO      : 0.05
  FREEZE_ENCODER    : False
  UNFREEZE_AT_EPOCH : 0
  EARLY_STOP_PATIENCE: 2
  MAX_LABEL_LENGTH  : 40


In [9]:
# 5. Metrics (dependency-free CER / WER / exact-match)
def _levenshtein(ref, hyp):
    m, n = len(ref), len(hyp)
    if m == 0: return n
    if n == 0: return m
    prev = list(range(n + 1))
    for i in range(1, m + 1):
        curr = [i] + [0] * n
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            curr[j] = min(prev[j]+1, curr[j-1]+1, prev[j-1]+cost)
        prev = curr
    return prev[n]

def compute_metrics(references, hypotheses):
    tce = tc = twe = tw = exact = 0
    for ref, hyp in zip(references, hypotheses):
        ref, hyp = str(ref), str(hyp)
        tce += _levenshtein(list(ref), list(hyp)); tc += len(ref)
        rw, hw = ref.split(), hyp.split()
        twe += _levenshtein(rw, hw); tw += len(rw)
        if ref == hyp: exact += 1
    n = len(references)
    return {'cer': (tce/tc) if tc else 0.0, 'wer': (twe/tw) if tw else 0.0,
            'accuracy': (exact/n) if n else 0.0, 'exact': exact, 'total': n}

def print_metrics(m, title='EVALUATION METRICS'):
    print('='*56); print(title); print('='*56)
    print(f"  Samples evaluated : {m['total']}")
    print(f"  Exact match       : {m['exact']}/{m['total']} ({m['accuracy']*100:.2f}%)")
    print(f"  CER               : {m['cer']*100:.2f}%")
    print(f"  WER               : {m['wer']*100:.2f}%")
    print('='*56)

print('[OK] Metrics')  # --status--

In [10]:
# 6. Load processor + model, APPLY THE DECODER-START FIX, sync generation config
assert os.path.exists(MANIFEST_CSV), f'manifest.csv not found at {MANIFEST_CSV} - set DATA_ROOT'

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
raw_model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

# TrOCR is pretrained to start decoding from the EOS/SEP token (id 2), NOT cls (id 0).
# Keep that convention and sync config + generation_config so train == inference.
raw_model.config.decoder_start_token_id = processor.tokenizer.sep_token_id
raw_model.config.eos_token_id = processor.tokenizer.sep_token_id
raw_model.config.pad_token_id = processor.tokenizer.pad_token_id
raw_model.config.vocab_size = raw_model.config.decoder.vocab_size
raw_model.generation_config.decoder_start_token_id = raw_model.config.decoder_start_token_id
raw_model.generation_config.eos_token_id = raw_model.config.eos_token_id
raw_model.generation_config.pad_token_id = raw_model.config.pad_token_id
raw_model.generation_config.max_length = cfg['MAX_LABEL_LENGTH']

raw_model.to(device)
print('Decoder start token id:', raw_model.config.decoder_start_token_id, '(should be 2)')
print('Params:', f"{sum(p.numel() for p in raw_model.parameters()):,}")

preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Decoder start token id: 2 (should be 2)
Params: 333,921,792


In [11]:
# 7. Dataset
class HandwrittenDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_len):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor
        self.max_len = max_len
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.img_dir, row['filename'])).convert('RGB')
        pixel_values = self.processor(images=image, return_tensors='pt').pixel_values.squeeze(0)
        labels = self.processor.tokenizer(str(row['label']).strip(), padding='max_length',
                                          max_length=self.max_len, truncation=True,
                                          return_tensors='pt').input_ids.squeeze(0)
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}

def load_split(split):
    df = pd.read_csv(MANIFEST_CSV, keep_default_na=False)
    df = df[df['split'] == split]
    df = df[df['label'].astype(str).str.strip() != '']
    df = df[df['label'].astype(str).str.upper() != 'UNREADABLE']
    return df.reset_index(drop=True)

train_df, val_df, test_df = load_split('train'), load_split('val'), load_split('test')
print(f'train={len(train_df)}  val={len(val_df)}  test={len(test_df)}')

train_ds = HandwrittenDataset(train_df, TRAIN_IMG_DIR, processor, cfg['MAX_LABEL_LENGTH'])
train_loader = DataLoader(train_ds, batch_size=cfg['BATCH_SIZE'], shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)

train=8000  val=1000  test=1000


In [12]:
# 8. Optimizer (separate LR for encoder/decoder), encoder freezing, scheduler, multi-GPU
encoder_params = list(raw_model.encoder.parameters())
decoder_params = [p for n, p in raw_model.named_parameters() if not n.startswith('encoder.')]
optimizer = AdamW([
    {'params': decoder_params, 'lr': cfg['DECODER_LR']},
    {'params': encoder_params, 'lr': cfg['ENCODER_LR']},
])

if cfg['FREEZE_ENCODER']:
    for p in raw_model.encoder.parameters():
        p.requires_grad = False
    print('Encoder FROZEN (will unfreeze at epoch', cfg['UNFREEZE_AT_EPOCH'], ')')

steps_per_epoch = max(1, len(train_loader) // cfg['GRAD_ACCUM_STEPS'])
total_steps = steps_per_epoch * cfg['EPOCHS']
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(total_steps * cfg['WARMUP_RATIO']), total_steps)

# Use both T4s for the training forward pass. Generation stays on raw_model (1 GPU).
model = raw_model
if n_gpu > 1:
    model = torch.nn.DataParallel(raw_model)
    print(f'Using DataParallel across {n_gpu} GPUs')

scaler = GradScaler('cuda')

print('[OK] Optimizer')  # --status--

Using DataParallel across 2 GPUs


In [13]:
# 9. Validation-CER evaluator (batched generation on a single GPU)
@torch.no_grad()
def evaluate_cer(df, img_dir, eval_model=None, batch_size=EVAL_BATCH_SIZE, show=False, title='METRICS'):
    mdl = eval_model if eval_model is not None else raw_model
    mdl.eval()
    refs, preds = [], []
    files = df['filename'].tolist()
    labels = df['label'].astype(str).tolist()
    for i in range(0, len(files), batch_size):
        chunk_f = files[i:i+batch_size]
        chunk_l = labels[i:i+batch_size]
        images = [Image.open(os.path.join(img_dir, f)).convert('RGB') for f in chunk_f]
        pv = processor(images=images, return_tensors='pt').pixel_values.to(device)
        gen = mdl.generate(pv, max_new_tokens=cfg['MAX_LABEL_LENGTH'])
        out = processor.batch_decode(gen, skip_special_tokens=True)
        preds.extend([o.strip() for o in out])
        refs.extend([l.strip() for l in chunk_l])
    m = compute_metrics(refs, preds)
    if show:
        print_metrics(m, title)
    return m

print('[OK] Validation')  # --status--

In [14]:
# 9b. BASELINE - evaluate the ORIGINAL base model on the TEST set (BEFORE fine-tuning)
# raw_model is still the unmodified base here (the training loop has not run yet).
# The decoder-start fix only re-affirms the base's own convention (id 2), so this is
# true base performance. Saved as `base_metrics` to compare against the fine-tuned model.
print('Evaluating BASE model (microsoft/trocr-base-handwritten) on the test set...')
base_metrics = evaluate_cer(test_df, TEST_IMG_DIR, show=True,
                            title='BASE MODEL - TEST METRICS (before fine-tuning)')

Evaluating BASE model (microsoft/trocr-base-handwritten) on the test set...
BASE MODEL - TEST METRICS (before fine-tuning)
  Samples evaluated : 1000
  Exact match       : 320/1000 (32.00%)
  CER               : 12.57%
  WER               : 52.98%


In [15]:
# 10. Training loop  --  AMP + grad accumulation + scheduled unfreeze + CER checkpointing + early stop
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_cer = float('inf')
patience = 0
start = time.time()

for epoch in range(1, cfg['EPOCHS'] + 1):
    if cfg['FREEZE_ENCODER'] and epoch == cfg['UNFREEZE_AT_EPOCH'] + 1:
        for p in raw_model.encoder.parameters():
            p.requires_grad = True
        print(f'  -> Encoder UNFROZEN at epoch {epoch}')

    model.train()
    running = 0.0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{cfg["EPOCHS"]}')
    for step, batch in enumerate(pbar, start=1):
        pv = batch['pixel_values'].to(device, non_blocking=True)
        lb = batch['labels'].to(device, non_blocking=True)
        with autocast('cuda'):
            out = model(pixel_values=pv, labels=lb)
            loss = out.loss.mean() / cfg['GRAD_ACCUM_STEPS']
        scaler.scale(loss).backward()
        running += loss.item() * cfg['GRAD_ACCUM_STEPS']
        if step % cfg['GRAD_ACCUM_STEPS'] == 0:
            scaler.unscale_(optimizer)  # unscale before clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
        pbar.set_postfix(loss=f'{running/step:.4f}', lr=f'{scheduler.get_last_lr()[0]:.2e}')

    val = evaluate_cer(val_df, VAL_IMG_DIR)
    print(f"  Epoch {epoch}: train_loss={running/len(train_loader):.4f}  "
          f"val_CER={val['cer']*100:.2f}%  val_acc={val['accuracy']*100:.2f}%")

    if val['cer'] < best_cer:
        best_cer = val['cer']
        raw_model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        patience = 0
        print(f'    saved best model (val_CER={best_cer*100:.2f}%)')
    else:
        patience += 1
        print(f'    no improvement ({patience}/{cfg["EARLY_STOP_PATIENCE"]})')
        if patience >= cfg['EARLY_STOP_PATIENCE']:
            print('    early stopping')
            break

print(f'\nDone in {(time.time()-start)/60:.1f} min. Best val CER: {best_cer*100:.2f}%')

Epoch 1/3:   0%|          | 0/500 [00:00<?, ?it/s]

  Epoch 1: train_loss=0.5980  val_CER=0.14%  val_acc=98.60%


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    saved best model (val_CER=0.14%)


Epoch 2/3:   0%|          | 0/500 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    saved best model (val_CER=0.13%)


Epoch 3/3:   0%|          | 0/500 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x790e365c1e40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x790e365c1e40>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

  Epoch 3: train_loss=0.0024  val_CER=0.14%  val_acc=98.80%
    no improvement (1/2)

Done in 24.9 min. Best val CER: 0.13%


In [16]:
# 11. Final evaluation of the BEST checkpoint on the locked TEST set
best = VisionEncoderDecoderModel.from_pretrained(OUTPUT_DIR).to(device)
test_metrics = evaluate_cer(test_df, TEST_IMG_DIR, eval_model=best, show=True,
                            title='FINE-TUNED MODEL - TEST METRICS')
print('\nModel saved at:', OUTPUT_DIR)
print('Download it from the Kaggle output panel (right side) after the run.')

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

FINE-TUNED MODEL - TEST METRICS
  Samples evaluated : 1000
  Exact match       : 986/1000 (98.60%)
  CER               : 0.20%
  WER               : 0.77%

Model saved at: /kaggle/working/trocr-finetuned
Download it from the Kaggle output panel (right side) after the run.


In [17]:
# 13. SIDE-BY-SIDE COMPARISON  (CER/WER: lower is better | Accuracy: higher is better)
def _p(x):
    return f'{x*100:.2f}%'
print('=' * 60)
print(f"{'Metric':<12}{'Base':>13}{'Fine-tuned':>15}{'Verdict':>14}")
print('-' * 60)
for key, name in [('cer', 'CER'), ('wer', 'WER'), ('accuracy', 'Accuracy')]:
    b, f = base_metrics[key], test_metrics[key]
    if key in ('cer', 'wer'):
        verdict = 'better' if f < b else ('same' if f == b else 'worse')
    else:
        verdict = 'better' if f > b else ('same' if f == b else 'worse')
    print(f'{name:<12}{_p(b):>13}{_p(f):>15}{verdict:>14}')
print('=' * 60)
print('Test samples:', base_metrics['total'])

Metric               Base     Fine-tuned       Verdict
------------------------------------------------------------
CER                12.57%          0.20%        better
WER                52.98%          0.77%        better
Accuracy           32.00%         98.60%        better
Test samples: 1000


In [18]:
import shutil
shutil.make_archive('/kaggle/working/trocr-finetuned', 'zip', '/kaggle/working/trocr-finetuned')
print('zipped -> /kaggle/working/trocr-finetuned.zip')

zipped -> /kaggle/working/trocr-finetuned.zip
